# Frente C: clasificación (etapas 10 y 11)

Clasificador por reglas AASM sobre los descriptores, desempate N1/REM con CSP y suavizado temporal por reglas. Sin datos reales usa `tabla_sintetica()`. Figuras con prefijo `c_`.

**Por qué no LDA + HMM.** El anteproyecto proponía LDA y un modelo oculto de Markov. La devolución aclara que en el TP cuatrimestral la clasificación no puede usar técnicas de inteligencia artificial. Ambos lo son: el LDA ajusta medias y covarianzas con épocas etiquetadas y el HMM aprende las transiciones de los hipnogramas de entrenamiento. Quedan como mejora para el final. En su lugar, cada umbral lo elige el diseñador mirando los histogramas de este notebook, y el suavizado usa reglas fijas de continuidad.

In [ ]:
# Entorno común a todos los notebooks: se corre desde codigo/ para que DIR_DATOS
# ("data" o $ESTADIOS_DATOS) apunte al mismo lugar que los scripts.
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
FIGURAS = Path.cwd().parent / "informe" / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt

from estadios.comun.config import Config
from estadios.comun.constantes import NOMBRES_ESTADIOS, N_ESTADIOS, SIN_ANOTAR
from estadios.datos.carga import ids_disponibles

USAR_DATOS_REALES = len(ids_disponibles()) > 0
print("Datos reales" if USAR_DATOS_REALES else "Sin datos descargados: se usan los sintéticos")

In [ ]:
def guardar(fig, nombre):
    """Guarda la figura en informe/figuras/ para incluirla con \\includegraphics."""
    fig.savefig(FIGURAS / f"{nombre}.pdf", bbox_inches="tight")

## Datos

Los umbrales se eligen **solo** con los pacientes de entrenamiento; los de prueba se usan una sola vez, al final.

In [ ]:
from dataclasses import replace

from estadios.comun.sintetico import tabla_sintetica
from estadios.datos.particion import particion_por_paciente
from estadios.pipeline import obtener_tabla

cfg = Config()
if USAR_DATOS_REALES:
    ids_entrenamiento, ids_prueba = particion_por_paciente(ids_disponibles())
    entrenamiento = [obtener_tabla(i, cfg) for i in ids_entrenamiento]
    prueba = [obtener_tabla(i, cfg) for i in ids_prueba]
else:
    entrenamiento = [tabla_sintetica(semilla=s) for s in range(8)]
    prueba = [tabla_sintetica(semilla=s) for s in range(100, 103)]
print(f"Entrenamiento: {len(entrenamiento)} registros. Prueba: {len(prueba)}.")

In [ ]:
y_entrenamiento = np.concatenate([t.estadios for t in entrenamiento])
anotadas = y_entrenamiento != SIN_ANOTAR
conteo = np.bincount(y_entrenamiento[anotadas], minlength=N_ESTADIOS)
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(NOMBRES_ESTADIOS, conteo / conteo.sum())
ax.set(ylabel="Fracción de épocas", title="Entrenamiento")
guardar(fig, "c_distribucion_estadios")

## Elección de los umbrales

Cada regla separa un estadio de los que más se le parecen. Para cada una se grafica el histograma del descriptor (en percentiles de su noche) en los dos grupos de épocas. La línea punteada marca dónde se cruzan los histogramas y la continua, el umbral actual de `ConfigClasificacion`. Si no coinciden, se decide mirando el gráfico y se copia el valor en `comun/config.py`.

In [ ]:
from estadios.clasificacion import reglas
from estadios.comun.constantes import N1, N2, N3, R, W

# Percentiles de todas las épocas de entrenamiento, con su estadio real.
por_tabla = [reglas.descriptores_de_reglas(t.matriz, t.nombres, cfg.clasificacion) for t in entrenamiento]
P = {k: np.concatenate([p[k] for p in por_tabla]) for k in por_tabla[0]}
Y = y_entrenamiento

c = cfg.clasificacion
TODOS = [W, N1, N2, N3, R]
# (descriptor, estadio que detecta, estadios de los que lo separa, campo del umbral)
UMBRALES = [
    ("emg",   [W],  [N1, N2, N3, R], "w_emg_min"),
    ("alfa",  [W],  [N1, N2, N3, R], "w_alfa_min"),
    ("delta", [N3], [N2],            "n3_delta_min"),
    ("ondas", [N3], [N2],            "n3_ondas_min"),
    ("emg",   [R],  [N1],            "r_emg_max"),
    ("eog",   [R],  [N1, N2],        "r_eog_min"),
    ("lf_hf", [R],  [N1, N2],        "r_lf_hf_min"),
    ("husos", [N2], [N1, R],         "n2_husos_min"),
]
UMBRALES = [u for u in UMBRALES if u[0] in P]

fig, axs = plt.subplots(2, 4, figsize=(14, 6))
filas = []
for ax, (clave, si, no, campo) in zip(axs.flat, UMBRALES):
    a, b = P[clave][np.isin(Y, si)], P[clave][np.isin(Y, no)]
    cruce = reglas.cruce_de_histogramas(a, b)
    actual = getattr(c, campo)
    bins = np.linspace(0, 100, 26)
    ax.hist(b, bins, density=True, alpha=0.5, label="/".join(NOMBRES_ESTADIOS[e] for e in no))
    ax.hist(a, bins, density=True, alpha=0.5, label="/".join(NOMBRES_ESTADIOS[e] for e in si))
    ax.axvline(cruce, ls="--", c="k")
    ax.axvline(actual, c="r")
    ax.set(title=campo, xlabel=f"Percentil de {clave}")
    ax.legend(fontsize=7)
    filas.append({"umbral": campo, "actual": actual, "cruce": round(cruce, 1)})
for ax in axs.flat[len(UMBRALES):]:
    ax.axis("off")
fig.tight_layout()
guardar(fig, "c_eleccion_umbrales")

import pandas as pd
pd.DataFrame(filas)

## Clasificación de los pacientes de prueba

Se comparan tres variantes del suavizado: ninguno, filtro de moda y reglas de continuidad.

In [ ]:
from estadios.clasificacion.modelo import ModeloEstadios
from estadios.evaluacion import metricas

y_real = np.concatenate([t.estadios for t in prueba])
modelos, reportes = {}, {}
for suavizado in ("ninguno", "moda", "reglas"):
    modelo = ModeloEstadios(replace(cfg.clasificacion, suavizado=suavizado)).ajustar(entrenamiento)
    y_pred = np.concatenate([modelo.predecir(t) for t in prueba])
    modelos[suavizado], reportes[suavizado] = modelo, metricas.reporte(y_real, y_pred)

resumen = pd.DataFrame({s: {"κ": r["kappa"], **r["f1"]} for s, r in reportes.items()}).T
resumen.index = ["Reglas", "Reglas + moda", "Reglas + continuidad"]
print(resumen.round(3).to_latex())  # Para pegar en informe/secciones/03_resultados.tex
resumen.round(3)

### Qué regla decidió cada época

Es lo que hace explicable al clasificador: cada época tiene una regla que la asignó. Se muestra cuántas épocas decidió cada regla y en qué fracción acertó (antes del suavizado).

In [ ]:
modelo = modelos["reglas"]
decisiones = [modelo.clasificar_epocas(t) for t in prueba]
y_reglas = np.concatenate([e for e, _ in decisiones])
regla = np.concatenate([r for _, r in decisiones])
validas = y_real != SIN_ANOTAR

uso = []
for i, nombre in enumerate(reglas.REGLAS):
    elegidas = validas & (regla == i)
    uso.append({"regla": nombre, "épocas": int(elegidas.sum()),
                "aciertos": float((y_reglas[elegidas] == y_real[elegidas]).mean()) if elegidas.any() else np.nan})
uso = pd.DataFrame(uso)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(uso["regla"], uso["épocas"])
for x, (n, acierto) in enumerate(zip(uso["épocas"], uso["aciertos"])):
    if n:
        ax.text(x, n, f"{acierto:.0%}", ha="center", va="bottom", fontsize=8)
ax.set(ylabel="Épocas de prueba", title="Regla que decidió cada época (acierto encima)")
guardar(fig, "c_uso_reglas")
uso

In [ ]:
def graficar_confusion(matriz, ax, titulo):
    normalizada = matriz / matriz.sum(axis=1, keepdims=True).clip(min=1)
    ax.imshow(normalizada, cmap="Blues", vmin=0, vmax=1)
    for i in range(N_ESTADIOS):
        for j in range(N_ESTADIOS):
            ax.text(j, i, f"{normalizada[i, j]:.2f}\n({matriz[i, j]})", ha="center", va="center", fontsize=7)
    ax.set_xticks(range(N_ESTADIOS), NOMBRES_ESTADIOS)
    ax.set_yticks(range(N_ESTADIOS), NOMBRES_ESTADIOS)
    ax.set(xlabel="Estimado", ylabel="Técnicos", title=titulo)

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
graficar_confusion(reportes["ninguno"]["confusion"], axs[0], "Reglas")
graficar_confusion(reportes["reglas"]["confusion"], axs[1], "Reglas + continuidad")
guardar(fig, "c_matrices_confusion")

### Hipnograma de un paciente de prueba

In [ ]:
ORDEN = {0: 4, 4: 3, 1: 2, 2: 1, 3: 0}  # W arriba, REM debajo, después N1, N2, N3

def a_altura(y):
    return np.array([ORDEN.get(int(e), np.nan) for e in y])

tabla = prueba[0]
horas = np.arange(len(tabla.estadios)) * 30 / 3600
fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True, sharey=True)
for ax, y, titulo in zip(axs,
                         [tabla.estadios, modelos["ninguno"].predecir(tabla), modelos["reglas"].predecir(tabla)],
                         ["Técnicos", "Reglas", "Reglas +\ncontinuidad"]):
    ax.step(horas, a_altura(y), where="post", lw=0.8)
    ax.set_ylabel(titulo)
axs[0].set_yticks([4, 3, 2, 1, 0], ["W", "R", "N1", "N2", "N3"])
axs[-1].set_xlabel("Tiempo (h)")
guardar(fig, f"c_hipnograma_{tabla.id}")

## κ por paciente

Se compara con el acuerdo entre técnicos (0,74) y con la red de referencia (0,79).

In [ ]:
kappas = {nombre: [metricas.kappa(t.estadios, modelos[s].predecir(t)) for t in prueba]
          for nombre, s in [("Reglas", "ninguno"), ("Reglas +\ncontinuidad", "reglas")]}
fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot(list(kappas.values()), tick_labels=list(kappas))
ax.axhline(0.74, ls="--", c="gray", label="Técnicos HMC")
ax.axhline(0.79, ls=":", c="gray", label="Red de referencia")
ax.set_ylabel("κ de Cohen")
ax.legend()
guardar(fig, "c_kappa_por_paciente")